# Task 3 - Agentic Workflows: Multi-Agent Financial Research System
CDAZZDEV Senior MLE Assessment

Runs Task 3A (single autonomous agent), Task 3B (two-agent coordination
with a critique loop), and Task 3C (memory + persistent cache +
observability) end-to-end. Cell outputs are kept (not cleared).

## 1. Setup

In [1]:
!pip install -q yfinance groq pydantic pandas numpy langchain-core langchain-groq langgraph duckduckgo-search

In [2]:
import os
# In Colab: click the key icon in the left sidebar -> add secret GROQ_API_KEY,
# then uncomment the two lines below. Locally, export it as an env var instead.
# from google.colab import userdata
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

assert os.environ.get("GROQ_API_KEY"), "Set GROQ_API_KEY before continuing (see cell above)."
TICKER = "AAPL"  # change this to research a different ticker

## 2. Task 3A - Single Autonomous Research Agent

In [3]:
from single_agent import run_single_agent

result_a = run_single_agent(TICKER)

print("--- MESSAGE TRACE (observe -> decide -> act) ---")
for step in result_a["trace"]:
    preview = (step["content"] or "")[:200]
    print(f"[{step['role']}] {preview}")
    if step["tool_calls"]:
        print("   tool_calls:", step["tool_calls"])

print("\n--- FINAL REPORT ---\n")
print(result_a["final_report"])

2026-10-06 13:58:20,698 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 13:58:26,293 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-10-06 13:58:27,598 [WARNING] Only 0 headlines retrieved for AAPL (target 10).
2026-10-06 13:58:28,225 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
d:\Machine_learning\MLOps\project\CDAZZDEV-MLE-MuhammedHasanMisba\task3_agentic\tools.py:163: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
2026-10-06 13:58:28,960 [INFO] response: https://www.bing.com/search?q=Apple+Inc.+latest+news+October+2026+share+price+outlook 200
2026-10-06 13:58:29,732 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
d:\Machine_learning\MLOps\project\CDAZZDEV-MLE-MuhammedHasanMisba\task3_agentic\tools.py:163: 

--- MESSAGE TRACE (observe -> decide -> act) ---
[human] Analyse the current financial health and market sentiment of AAPL. Identify the top three risks to its share price over the next 90 days and suggest one data-driven hedge strategy.
[ai] 
   tool_calls: [{'name': 'get_price_data', 'args': {'period': '2y', 'ticker': 'AAPL'}, 'id': 'fc_79dd1464-0743-4bc7-a823-d86e5cd644c3', 'type': 'tool_call'}]
[tool] {"ticker": "AAPL", "latest_close": 332.89, "sma_50": 322.3583, "sma_200": 289.0941, "rsi_14": 53.8071, "macd": 3.4961, "macd_signal": 4.5703}
[ai] 
   tool_calls: [{'name': 'get_news', 'args': {'n': 10, 'ticker': 'AAPL'}, 'id': 'fc_9530580c-3321-42ca-b7d3-1f63a91c4e52', 'type': 'tool_call'}]
[tool] {"ticker": "AAPL", "headlines": []}
[ai] 
   tool_calls: [{'name': 'web_search', 'args': {'max_results': 10, 'query': 'Apple Inc. latest news October 2026 share price outlook'}, 'id': 'fc_62c6fc54-d8db-4ae2-b537-730ef1a18c67', 'type': 'tool_call'}]
[tool] {"query": "Apple Inc. latest news O

**Check:** the tool-call order above should differ sensibly if you change
`TICKER` -- that's the evidence tool selection is genuinely autonomous, not
a fixed sequence.

## 3. Task 3B - Two-Agent Coordination (Data Analyst + Research Writer)

In [4]:
from multi_agent import run_multi_agent
import json

result_b = run_multi_agent(TICKER)

print("--- AGENT MESSAGE / HANDOFF TRACE ---")
print(json.dumps(result_b["trace"], indent=2, default=str))

print("\n--- STRUCTURED DATA BRIEF (Pydantic handoff, Agent A -> Agent B) ---")
print(json.dumps(result_b["data_brief"], indent=2, default=str))

print("\n--- FINAL REPORT ---\n")
print(result_b["final_report"])

2026-10-06 14:00:57,361 [WARNING] Only 0 headlines retrieved for AAPL (target 10).
d:\Machine_learning\MLOps\project\CDAZZDEV-MLE-MuhammedHasanMisba\task3_agentic\tools.py:163: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
2026-10-06 14:00:57,919 [INFO] response: https://www.bing.com/search?q=AAPL+stock+analyst+outlook+risks 200
2026-10-06 14:01:00,498 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


--- AGENT MESSAGE / HANDOFF TRACE ---
[
  {
    "agent": "DataAnalyst",
    "action": "build_brief",
    "output": {
      "ticker": "AAPL",
      "latest_close": 332.89,
      "sma_50": 322.3583,
      "sma_200": 289.0941,
      "rsi_14": 53.8071,
      "macd": 3.4961,
      "macd_signal": 4.5703,
      "annualised_volatility_pct": 21.12,
      "price_regime_note": "Quant regime: trend-up",
      "clarification_answer": null
    }
  },
  {
    "agent": "ResearchWriter",
    "action": "gather_qualitative_context",
    "output": {
      "n_news": 0,
      "n_search_results": 5
    }
  },
  {
    "agent": "ResearchWriter",
    "action": "write_report",
    "output_preview": "### Financial Health Summary\n- **Price Momentum:** The stock is trading at **$332.89**, comfortably above its **50\u2011day SMA ($322.36)** and **200\u2011day SMA ($289.09)**, confirming the \u201ctrend\u2011up\u201d regime n"
  }
]

--- STRUCTURED DATA BRIEF (Pydantic handoff, Agent A -> Agent B) ---
{
  "ticker": 

**Check:** the trace above should contain a `request_clarification` step
followed by an `answer_clarification` step -- that's the required critique
loop executing end-to-end with no manual intervention.

## 4. Task 3C - Short-Term Memory, Persistent Cache, Observability

In [5]:
from memory_observability import ResearchSession, print_trace_log_summary

session = ResearchSession()

print("=== Run 1: cold (executes agents/tools) ===")
r1 = session.research(TICKER)
print("source:", r1["source"])

print("\n=== Run 2: same ticker, same session (short-term memory) ===")
r2 = session.research(TICKER)
print("source:", r2["source"])

print("\n=== Follow-up question answered from memory, zero tool calls ===")
print(session.ask_followup(TICKER, "How volatile has it been recently?"))

=== Run 1: cold (executes agents/tools) ===


2026-10-06 14:01:07,702 [WARNING] Only 0 headlines retrieved for AAPL (target 10).
d:\Machine_learning\MLOps\project\CDAZZDEV-MLE-MuhammedHasanMisba\task3_agentic\tools.py:163: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
2026-10-06 14:01:08,208 [INFO] response: https://www.bing.com/search?q=AAPL+stock+analyst+outlook+risks 200
2026-10-06 14:01:10,370 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


source: live_run

=== Run 2: same ticker, same session (short-term memory) ===
source: short_term_memory

=== Follow-up question answered from memory, zero tool calls ===
(answered from session memory, no tool calls) For AAPL: RSI(14)=53.8071, volatility=21.12%, in response to: 'How volatile has it been recently?'


In [6]:
# New session object (simulates a fresh run/process on the same day) --
# should hit the persistent disk cache instead of re-running everything.
session2 = ResearchSession()
r3 = session2.research(TICKER)
print("new-session source:", r3["source"])

new-session source: persistent_cache


In [7]:
print_trace_log_summary()

22 tool calls logged to d:\Machine_learning\MLOps\project\CDAZZDEV-MLE-MuhammedHasanMisba\task3_agentic\logs\agent_trace.jsonl
  web_search               704.9 ms  {"query": "Apple stock outlook", "results": [{"title": "Apple", "snippet": "Disc
  web_search               620.2 ms  {"query": "Apple supply chain issues 2026", "results": [{"title": "Apple", "snip
  get_price_data          1232.2 ms  {"ticker": "AAPL", "latest_close": 332.89, "sma_50": 322.3583, "sma_200": 289.09
  calculate_volatility     184.1 ms  {"ticker": "AAPL", "window": 30, "annualised_volatility_pct": 21.12}
  get_news                 946.3 ms  {"ticker": "AAPL", "headlines": []}
  web_search               712.2 ms  {"query": "AAPL stock analyst outlook risks", "results": [{"title": "WhatsApp We
  get_price_data           341.4 ms  {"ticker": "AAPL", "latest_close": 332.89, "sma_50": 322.3583, "sma_200": 289.09
  calculate_volatility     146.2 ms  {"ticker": "AAPL", "window": 30, "annualised_volatility_pct": 21.12

**Check:** `task3_agentic/logs/agent_trace.jsonl` should now contain one
JSON line per tool call across all three sections above (name, inputs,
truncated output, duration in ms) -- this file is a required Task 3
deliverable and must be committed to the repository.